# 💻 Módulo 01 - Práctica: MLOps con MLflow

## Ejercicios de Tracking y Registry

```python
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

# Generar datos sintéticos
np.random.seed(42)
n_samples = 1000
X = np.random.randn(n_samples, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(✅" Datos preparados: {} train, {} test".format(len(X_train), len(X_test)))

# Ejercicio 1: Tracking de experimentos
mlflow.set_experiment("mlops_practice")

print("\n🧪 Ejercicio 1: Experimentación con hiperparámetros")

results = []

for n_estimators in [50, 100, 200]:
    for max_depth in [5, 10, None]:
        with mlflow.start_run(run_name=f"rf_n{n_estimators}_d{max_depth}"):
            # Entrenar modelo
            model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=42
            )
            model.fit(X_train, y_train)
            
            # Predecir
            y_pred = model.predict(X_test)
            
            # Calcular métricas
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            # Log parameters
            mlflow.log_param("n_estimators", n_estimators)
            mlflow.log_param("max_depth", max_depth if max_depth else "None")
            mlflow.log_param("random_state", 42)
            
            # Log metrics
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
            # Log model
            mlflow.sklearn.log_model(model, "random_forest_model")
            
            # Tags
            mlflow.set_tag("model_type", "RandomForest")
            mlflow.set_tag("task", "classification")
            
            results.append({
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "accuracy": accuracy,
                "f1_score": f1
            })
            
            print(f"  n_estimators={n_estimators}, max_depth={max_depth}: "
                  f"accuracy={accuracy:.4f}, f1={f1:.4f}")

# Ejercicio 2: Comparación de experimentos
print("\n📊 Ejercicio 2: Comparación de resultados")

df_results = pd.DataFrame(results)
best_run = df_results.loc[df_results['accuracy'].idxmax()]

print("\nMejor configuración:")
print(f"  n_estimators: {best_run['n_estimators']}")
print(f"  max_depth: {best_run['max_depth']}")
print(f"  accuracy: {best_run['accuracy']:.4f}")
print(f"  f1_score: {best_run['f1_score']:.4f}")

# Ejercicio 3: Registry de modelos
print("\n📋 Ejercicio 3: Model Registry")

# Buscar el mejor run
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("mlops_practice")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)

if runs:
    best_run_id = runs[0].info.run_id
    best_accuracy = runs[0].data.metrics["accuracy"]
    
    print(f"✅ Mejor run encontrado: {best_run_id[:8]}... (accuracy: {best_accuracy:.4f})")
    
    # Registrar modelo
    model_name = "rf_classifier_practice"
    model_uri = f"runs:/{best_run_id}/random_forest_model"
    
    try:
        registered_model = mlflow.register_model(model_uri, model_name)
        print(f"✅ Modelo registrado: {model_name} v{registered_model.version}")
    except Exception as e:
        print(f"⚠️ Ya existe un modelo con ese nombre: {e}")
else:
    print("⚠️ No se encontraron runs")

# Ejercicio 4: Simulación de CI/CD
print("\n🔄 Ejercicio 4: Validación de modelo (CI)")

# Test de smoke (validaciones básicas)
def smoke_test(model, X_test, y_test):
    """Validaciones básicas de un modelo."""
    tests_passed = []
    
    # Test 1: Modelo puede predecir
    try:
        predictions = model.predict(X_test)
        tests_passed.append(("✅", "Modelo puede predecir"))
    except Exception as e:
        tests_passed.append(("❌", f"Modelo falla al predecir: {e}"))
        return tests_passed
    
    # Test 2: Predicciones en rango válido
    if set(predictions).issubset({0, 1}):
        tests_passed.append(("✅", "Predicciones en rango válido [0, 1]"))
    else:
        tests_passed.append(("❌", "Predicciones fuera de rango"))
    
    # Test 3: Accuracy mínima
    accuracy = accuracy_score(y_test, predictions)
    if accuracy >= 0.75:
        tests_passed.append(("✅", f"Accuracy >= 0.75 ({accuracy:.4f})"))
    else:
        tests_passed.append(("❌", f"Accuracy < 0.75 ({accuracy:.4f})"))
    
    # Test 4: Sin predicciones constantes
    if len(set(predictions)) > 1:
        tests_passed.append(("✅", "Modelo no predice constante"))
    else:
        tests_passed.append(("❌", "Modelo predice siempre la misma clase"))
    
    return tests_passed

# Ejecutar smoke test en el mejor modelo
if runs:
    model_uri = f"runs:/{best_run_id}/random_forest_model"
    loaded_model = mlflow.sklearn.load_model(model_uri)
    
    test_results = smoke_test(loaded_model, X_test, y_test)
    
    print("\nResultados de Smoke Tests:")
    for status, message in test_results:
        print(f"  {status} {message}")
    
    all_passed = all(status == "✅" for status, _ in test_results)
    if all_passed:
        print("\n🎉 Todos los tests pasados - Modelo listo para deployment")
    else:
        print("\n⚠️ Algunos tests fallaron - Revisar modelo")

print("\n✅ Ejercicios de MLOps completados")
print("💡 En producción, estos pasos estarían automatizados con CI/CD")
print("💡 MLflow UI: Ver experimentos y modelos en la pestaña Experiments")
```

---

**Universidad del Aconcagua 🇦🇷**